In [16]:
# Imports
import sys
import os
import io
sys.path.append('..')

import numpy as np
import pandas as pd

# Internal
from pipeline.libs.src.utils import get_project_name
from pipeline.libs.src.aws import get_serialized_from_S3

In [17]:
# Setup Logging
import logging
# Configure root logger
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(name)s: %(message)s'
)
logging.getLogger("pipeline.libs.src.aws").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

In [18]:
# import dataset
# Read dataset from S3 (assume existing objet)
path_to_data = f'data/transformed/df_cars_transformed.pkl'
logger.info(f"Path to data to extract relatively to project root: {path_to_data}")

# project_root_path defined by PROJECT_NAME in .env
project_root_path = get_project_name()

# Deserialize from Bucket/Project/...
s3_key = f'{project_root_path}/{path_to_data}'
logger.info(f"S3 key relative to bucket root: {s3_key}")

# Load from S3: S3_key = file path relative to bucket name defined in ENV
dataset = get_serialized_from_S3(s3_key)
dataset.head()

[INFO] __main__: Path to data to extract relatively to project root: data/transformed/df_cars_transformed.pkl
[INFO] __main__: S3 key relative to bucket root: ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Deserializing the file from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: File successfully loaded from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Type of Object deserialized from S3: <class 'pandas.core.frame.DataFrame'>
[INFO] pipeline.libs.src.aws: Object successfully deserialized from S3.


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131


---
# Separate Features/Target

In [19]:
# Separate X/Y
target = "rental_price_per_day"
features = [col for col in dataset.columns if col != target]

X = dataset.loc[:, features].copy()
Y = dataset.loc[:, target].copy()
print(f'X.shape: {X.shape}')
print(f'Y.shape: {Y.shape}')

X.shape: (4747, 13)
Y.shape: (4747,)


---
# Train test split

In [20]:
# train/test split
from sklearn.model_selection import train_test_split
print('train and test split')
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.18, random_state=74)
print('✅Done')
print('=' * 80)
print(f'X_train {X_train.shape}:\n{X_train.head(3)}')
print('-' * 80)
print(f'X_test {X_test.shape}:\n{X_test.head(3)}')
print('=' * 80)
print(f'Y_train:\n{Y_train.shape}\n{Y_train}')
print('-' * 80)
print(f'Y_test:\n{Y_test.shape}\n{Y_test}')

train and test split
✅Done
X_train (3892, 13):
     model_key  mileage  engine_power    fuel paint_color car_type  \
708       Audi   192114           160  diesel       white   estate   
1641   Peugeot   132180           100  diesel      silver   estate   
3440      Audi   179459           150  diesel       black    sedan   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
708                        True     True                  True           True   
1641                       True     True                 False          False   
3440                       True     True                  True          False   

      has_getaround_connect  has_speed_regulator  winter_tires  
708                    True                False          True  
1641                  False                False          True  
3440                   True                 True          True  
--------------------------------------------------------------------------------
X_test

---
# Basic Pipeline

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Separate features by type
l_num_features = ["mileage", "engine_power"]
l_cat_features = [col for col in features if col not in l_num_features]

# Pipelines 
# Pipeline for numeric columns
pipeline_num_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy="mean")),
        ('scaler', StandardScaler())
        ]
    )
print('Numeric transformer:')
print(pipeline_num_transformer)

# Pipeline for categorical columns
pipeline_cat_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(drop="first", handle_unknown='ignore'))
        ]
    )
print('Categorical transformer:')
print(pipeline_cat_transformer)

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", pipeline_num_transformer, l_num_features),
        ("cat", pipeline_cat_transformer, l_cat_features)
        ]
    )


Numeric transformer:
Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])
Categorical transformer:
Pipeline(steps=[('encoder',
                 OneHotEncoder(drop='first', handle_unknown='ignore'))])


---
# Fit transform

In [22]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

---
# Baseline

In [23]:
# train model
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train, Y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [24]:
# Predicts
from sklearn.metrics import r2_score

Y_train_pred = model_lr.predict(X_train)
Y_test_pred = model_lr.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.7132526748752186
R2 score on test set :  0.7431348016517181


Le score sur le jeu de test est supérieur à celui du train : le modèle généralise bien, voire un peu mieux sur ce sous-échantillon

In [25]:
# Visualize best coefficients
import plotly.express as px

l_columns = []
for name, pipeline, features_list in preprocessor.transformers_: # for each tuple
    if name == 'num': 
        l_features = features_list 
    else: 
        l_features = pipeline.named_steps['encoder'].get_feature_names_out()
        l_features = list(l_features)
        l_features = np.char.replace(l_features, 'x0', features_list[0])
        l_features = np.char.replace(l_features, 'x1', features_list[1])
    l_columns.extend(l_features) 
        
coefs = pd.DataFrame(
    index=l_columns,
    data=model_lr.coef_.transpose(),
    columns=["coefficients"]
    )

features_importance = coefs.sort_values(by='coefficients', key=abs)

fig = px.bar(
    features_importance,
    orientation='h',
    height=700
    )
fig.update_layout(
    showlegend=False, 
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

---
# Random Forest

In [26]:
# Import
from sklearn.ensemble import RandomForestRegressor

model_rfr = RandomForestRegressor()
model_rfr.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
Y_train_pred = model_rfr.predict(X_train)
Y_test_pred = model_rfr.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.9667663810498689
R2 score on test set :  0.7642676345093794


Overfitting

In [28]:
# Reduce overfitting - GridsearchCV
from sklearn.model_selection import GridSearchCV

d_params = {
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [2, 4, 6],
    'min_samples_split': [2, 4, 6],
    'n_estimators': [10, 50, 100, 200, 300]
    }

gs = GridSearchCV(
        estimator=model_rfr,
        param_grid=d_params,
        cv=3,
        verbose=2
        )
gs.fit(X_train, Y_train)

Fitting 3 folds for each of 135 candidates, totalling 405 fits
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.8s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.8s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.8s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   1.7s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   1.7s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   1.6s
[CV] END max_depth=10, min_samples

,estimator,RandomForestRegressor()
,param_grid,"{'max_depth': [10, 15, ...], 'min_samples_leaf': [2, 4, ...], 'min_samples_split': [2, 4, ...], 'n_estimators': [10, 50, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [29]:
# Predicts
print("Best Parameters:", gs.best_params_)
print("Best Score:", gs.best_score_)

Y_train_pred = gs.predict(X_train)
Y_test_pred = gs.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Best Parameters: {'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best Score: 0.7470208156012849
R2 score on training set :  0.9197027070404109
R2 score on test set :  0.7634322569277923


---
# AdaBoost

In [30]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

base = DecisionTreeRegressor(max_depth=3)

model_abr = AdaBoostRegressor(estimator=base)
d_params = {
    'learning_rate': [0.05, 0.5, 1.0, 5.0, 10],
    'loss': ['linear', 'square', 'exponential'],
    'n_estimators': [10, 60, 130, 220]
    }
gs_abr = GridSearchCV(
    model_abr,
    param_grid=d_params,
    cv=3,
    verbose=2
    )
gs_abr.fit(X_train, Y_train)

Fitting 3 folds for each of 60 candidates, totalling 180 fits
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=10; total time=   0.1s
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=10; total time=   0.0s
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=10; total time=   0.0s
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=60; total time=   0.2s
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=60; total time=   0.2s
[CV] END ...learning_rate=0.05, loss=linear, n_estimators=60; total time=   0.2s
[CV] END ..learning_rate=0.05, loss=linear, n_estimators=130; total time=   0.5s
[CV] END ..learning_rate=0.05, loss=linear, n_estimators=130; total time=   0.5s
[CV] END ..learning_rate=0.05, loss=linear, n_estimators=130; total time=   0.5s
[CV] END ..learning_rate=0.05, loss=linear, n_estimators=220; total time=   0.8s
[CV] END ..learning_rate=0.05, loss=linear, n_estimators=220; total time=   0.9s
[CV] END ..learning_rate=0.05, loss=linear, n_e

,estimator,AdaBoostRegre...(max_depth=3))
,param_grid,"{'learning_rate': [0.05, 0.5, ...], 'loss': ['linear', 'square', ...], 'n_estimators': [10, 60, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [31]:

print("Best Parameters:", gs_abr.best_params_)
print("Best Score:", gs_abr.best_score_)

Y_train_pred = gs_abr.predict(X_train)
Y_test_pred = gs_abr.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Best Parameters: {'learning_rate': 0.05, 'loss': 'exponential', 'n_estimators': 130}
Best Score: 0.5956311601406258
R2 score on training set :  0.6182858925547243
R2 score on test set :  0.6084355598395308


le modèle n'apprend pas bien

---
# XGBoost

In [47]:
!pip install --upgrade pip
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 559.5 kB/s  0:00:03 eta 0:00:02
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


In [32]:
from xgboost import XGBRegressor

model_xgbr = XGBRegressor()
d_params = {
    'max_depth': [2, 4, 6, 8],
    "min_child_weight": [10, 15, 20],
    "n_estimators": [125, 150, 175, 200],
    "learning_rate": [0.05, 0.1]
    }

gs_xgbr = GridSearchCV(
    model_xgbr,
    param_grid=d_params,
    cv=3, 
    verbose=2
    )

gs_xgbr.fit(X_train, Y_train)

Fitting 3 folds for each of 96 candidates, totalling 288 fits
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=125; total time=   0.4s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=125; total time=   0.4s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=125; total time=   0.2s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=150; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=150; total time=   0.3s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=150; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=175; total time=   0.1s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=175; total time=   0.2s
[CV] END learning_rate=0.05, max_depth=2, min_child_weight=10, n_estimators=175; total time=   0.1s
[CV] END learning_rate=0.05, max_depth

,estimator,"XGBRegressor(...ree=None, ...)"
,param_grid,"{'learning_rate': [0.05, 0.1], 'max_depth': [2, 4, ...], 'min_child_weight': [10, 15, ...], 'n_estimators': [125, 150, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [33]:
# Predicts
print("Best Parameters:", gs_xgbr.best_params_)
print("Best Score:", gs_xgbr.best_score_)

Y_train_pred = gs_xgbr.predict(X_train)
Y_test_pred = gs_xgbr.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Best Parameters: {'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 10, 'n_estimators': 175}
Best Score: 0.7627555131912231
R2 score on training set :  0.8826720714569092
R2 score on test set :  0.789702296257019


overfitting de 0.1 - le modèle sur-apprend.

Conclusion: il faut modifier le dataset pour diminuer la complexité de certaines catégories via du feature engineering

---
---
# Feature engineering for cardinality lowering
---

Y a-t-il des catégories rares qui pourraient être regroupées ?

In [89]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_cars = dataset.copy()
# Display proportion of each category in every categorical column
# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars.select_dtypes(include=["object", "category"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution",
    height=440 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=10, b=10)  # left, right, top, bottom
)
fig.show()

Approche = réduire la cardinalité en regroupant les valeurs par catégorie et par proximité de valeur des coefficients issus de la baseline, pour ne pas perdre une information intéressante même sur une catégorie minoritaire (ex: couleur verte, ou car_type=van)

Extraire le nom de la colonne d'origine ainsi que sa value

In [90]:
l_columns = []
for name, pipeline, features_list in preprocessor.transformers_: # for each tuple
    if name == 'num': 
        l_features = features_list 
    else: 
        l_features = pipeline.named_steps['encoder'].get_feature_names_out()
        l_features = list(l_features)
        l_features = np.char.replace(l_features, 'x0', features_list[0])
        l_features = np.char.replace(l_features, 'x1', features_list[1])
    l_columns.extend(l_features) 
        
coefs = pd.DataFrame(
    index=l_columns,
    data=model_le.coef_.transpose(),
    columns=["coefficients"]
    )

df_features_importance = coefs.sort_values(by='coefficients', key=abs)

# Get original feature's name to associate with coefficient
def add_nature_column(df_features_importance, original_columns):
    """
    Ajoute une colonne 'nature' indiquant le nom de la colonne d'origine
    pour chaque feature encodée ou non.
    
    Paramètres :
        df_features_importance : DataFrame dont l'index contient les noms des features.
        original_columns : liste des noms des colonnes d'origine (catégorielles et numériques).
    """
    def find_origin(feature_name):
        for col in original_columns:
            if feature_name.startswith(col):
                return col
        return None  # if no matching

    df = df_features_importance.copy()
    df["nature"] = df.index.map(find_origin)
    return df

# List of original columns
l_original_columns = df_cars.columns.tolist()
# Create column original feature's name 
df_features_importance = add_nature_column(df_features_importance, l_original_columns)

# Get original feature's value to associate with coefficient
def add_category_value(df_features_importance, l_categorical_cols):
    """
    Pour les colonnes catégorielles, extrait la valeur de la modalité
    depuis l'index en retirant le préfixe correspondant à 'nature'.
    """
    def extract_value(row):
        if row['nature'] in l_categorical_cols:
            prefix = row['nature'] + "_"
            if row.name.startswith(prefix):
                return row.name[len(prefix):]
        return None  # ou '' si tu préfères
    
    df = df_features_importance.copy()
    df['value'] = df.apply(extract_value, axis=1)
    return df

# Create column original feature's value
df_features_importance = add_category_value(df_features_importance, l_categorical_cols)

display(df_features_importance)

,coefficients,nature,value
paint_color_black,0.375681,paint_color,black
has_air_conditioning_True,0.445991,has_air_conditioning,None
paint_color_brown,0.801588,paint_color,brown
car_type_coupe,0.847586,car_type,coupe
private_parking_available_True,0.992997,private_parking_available,None
model_key_Subaru,1.225058,model_key,Subaru
paint_color_red,1.394943,paint_color,red
paint_color_grey,-1.489456,paint_color,grey
car_type_sedan,-2.906554,car_type,sedan
model_key_Peugeot,-3.168598,model_key,Peugeot


Créer des clusters de values dont les coefficients sont proches. On modifie max_distance pour affiner

In [122]:
from sklearn.cluster import AgglomerativeClustering
import numpy as np

def cluster_hierarchical(df, nature_col='nature', coef_col='coefficients', value_col='value', max_distance=10): # Modify max_distance to adapt cluster
    df = df.copy()
    df['cluster'] = None

    for nature, group in df[df[value_col].notnull()].groupby(nature_col):
        X = group[coef_col].values.reshape(-1,1)
        if len(X) == 1:
            df.loc[group.index, 'cluster'] = f"{nature}_cluster1"
            continue

        clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=max_distance, linkage='ward')
        labels = clustering.fit_predict(X)
        for label in np.unique(labels):
            idx = group.index[labels == label]
            df.loc[idx, 'cluster'] = f"{nature}_cluster{label+1}"
    return df

df_features_importance = cluster_hierarchical(df_features_importance)
print(df_features_importance["cluster"].nunique())
display(df_features_importance)

10


,coefficients,nature,value,cluster
paint_color_black,0.375681,paint_color,black,paint_color_cluster1
has_air_conditioning_True,0.445991,has_air_conditioning,None,None
paint_color_brown,0.801588,paint_color,brown,paint_color_cluster1
car_type_coupe,0.847586,car_type,coupe,car_type_cluster1
private_parking_available_True,0.992997,private_parking_available,None,None
model_key_Subaru,1.225058,model_key,Subaru,model_key_cluster3
paint_color_red,1.394943,paint_color,red,paint_color_cluster1
paint_color_grey,-1.489456,paint_color,grey,paint_color_cluster1
car_type_sedan,-2.906554,car_type,sedan,car_type_cluster1
model_key_Peugeot,-3.168598,model_key,Peugeot,model_key_cluster2


In [123]:
# renommer chaque cluster pour qu’il contienne la liste des valeurs catégorielles correspondantes, plutôt que _cluster1, _cluster2,
def rename_clusters_with_values(df):
    df = df.copy()
    
    # On parcourt toutes les colonnes catégorielles et clusters existants
    for nature in df['nature'].dropna().unique():
        # on sélectionne uniquement les lignes de cette nature
        df_nature = df[(df['nature'] == nature) & df['cluster'].notnull()]
        
        for cluster in df_nature['cluster'].unique():
            # récupérer toutes les valeurs correspondant à ce cluster
            values = df_nature[df_nature['cluster'] == cluster]['value'].tolist()
            values_str = "-".join(values)
            
            # créer le nouveau nom de cluster
            new_cluster_name = f"{nature}_cluster[{values_str}]"
            
            # remplacer dans le dataframe
            df.loc[df_nature[df_nature['cluster'] == cluster].index, 'cluster'] = new_cluster_name
    
    return df

df_features_importance = rename_clusters_with_values(df_features_importance)
display(df_features_importance)

,coefficients,nature,value,cluster
paint_color_black,0.375681,paint_color,black,paint_color_cluster[black-brown-red-grey-white...
has_air_conditioning_True,0.445991,has_air_conditioning,None,None
paint_color_brown,0.801588,paint_color,brown,paint_color_cluster[black-brown-red-grey-white...
car_type_coupe,0.847586,car_type,coupe,car_type_cluster[coupe-sedan-suv]
private_parking_available_True,0.992997,private_parking_available,None,None
model_key_Subaru,1.225058,model_key,Subaru,model_key_cluster[Subaru-Renault-Maserati-Ferr...
paint_color_red,1.394943,paint_color,red,paint_color_cluster[black-brown-red-grey-white...
paint_color_grey,-1.489456,paint_color,grey,paint_color_cluster[black-brown-red-grey-white...
car_type_sedan,-2.906554,car_type,sedan,car_type_cluster[coupe-sedan-suv]
model_key_Peugeot,-3.168598,model_key,Peugeot,model_key_cluster[Peugeot-BMW-Citroën]


mapping des valeurs originales de df_cars vers les clusters calculés dans df_features_importance, par nature et value

In [124]:
# Créer le dictionnaire de mapping
# structure : { 'brand': { 'BMW': 'brand_cluster[BMW]', ... }, 'color': { ... } }
import re

# Extraire uniquement la partie entre crochets
def extract_bracket(s):
    match = re.search(r'\[(.*)\]', s)
    return match.group(1) if match else s

# Créer le dictionnaire de mapping avec uniquement le contenu entre crochets
mapping = {}
df_map = df_features_importance[df_features_importance['value'].notnull()]
for nature, group in df_map.groupby('nature'):
    mapping[nature] = {v: extract_bracket(c) for v, c in zip(group['value'], group['cluster'])}

# Appliquer le mapping dans df_cars
df_cars_clustered = df_cars.copy()

for col in mapping.keys():
    if col in df_cars_clustered.columns:
        df_cars_clustered[col] = df_cars_clustered[col].map(mapping[col]).fillna(df_cars_clustered[col])

display(df_cars_clustered)

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Peugeot-BMW-Citroën,140411,100,diesel,black-brown-red-grey-white-blue-silver,convertible,True,True,False,False,True,True,True,106
2,Peugeot-BMW-Citroën,183297,120,diesel,black-brown-red-grey-white-blue-silver,convertible,False,False,False,False,True,False,True,101
3,Peugeot-BMW-Citroën,128035,135,diesel,black-brown-red-grey-white-blue-silver,convertible,True,True,False,False,True,True,True,158
4,Peugeot-BMW-Citroën,97097,160,diesel,black-brown-red-grey-white-blue-silver,convertible,True,True,False,False,False,True,True,183
5,Peugeot-BMW-Citroën,152352,225,petrol,black-brown-red-grey-white-blue-silver,convertible,True,True,False,False,True,True,True,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Mercedes-Volkswagen-Opel-Toyota-SEAT,39743,110,diesel,black-brown-red-grey-white-blue-silver,van,False,True,False,False,False,False,True,121
4839,Mercedes-Volkswagen-Opel-Toyota-SEAT,49832,100,diesel,black-brown-red-grey-white-blue-silver,van,False,True,False,False,False,False,True,132
4840,Mercedes-Volkswagen-Opel-Toyota-SEAT,19633,110,diesel,black-brown-red-grey-white-blue-silver,van,False,True,False,False,False,False,True,130
4841,Mercedes-Volkswagen-Opel-Toyota-SEAT,27920,110,diesel,black-brown-red-grey-white-blue-silver,van,True,True,False,False,False,False,True,151


In [126]:
# Display proportion of each category in every categorical column
# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars_clustered.select_dtypes(include=["object", "category"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars_clustered[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution",
    height=440 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=10, b=10)  # left, right, top, bottom
)
fig.show()

Relancer une baseline sur ce nouveau dataset

---
Separate Features/Target

In [127]:
# Separate X/Y
target = "rental_price_per_day"
features = [col for col in df_cars_clustered.columns if col != target]

X = df_cars_clustered.loc[:, features].copy()
Y = df_cars_clustered.loc[:, target].copy()
print(f'X.shape: {X.shape}')
print(f'Y.shape: {Y.shape}')

X.shape: (4747, 13)
Y.shape: (4747,)


---
Train test split

In [128]:
# train/test split
print('train and test split')
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.18, random_state=74)
print('✅Done')
print('=' * 80)
print(f'X_train {X_train.shape}:\n{X_train.head(3)}')
print('-' * 80)
print(f'X_test {X_test.shape}:\n{X_test.head(3)}')
print('=' * 80)
print(f'Y_train:\n{Y_train.shape}\n{Y_train}')
print('-' * 80)
print(f'Y_test:\n{Y_test.shape}\n{Y_test}')

train and test split
✅Done
X_train (3892, 13):
                model_key  mileage  engine_power    fuel  \
708                  Audi   192114           160  diesel   
1641  Peugeot-BMW-Citroën   132180           100  diesel   
3440                 Audi   179459           150  diesel   

                                 paint_color                     car_type  \
708   black-brown-red-grey-white-blue-silver  subcompact-hatchback-estate   
1641  black-brown-red-grey-white-blue-silver  subcompact-hatchback-estate   
3440  black-brown-red-grey-white-blue-silver              coupe-sedan-suv   

      private_parking_available  has_gps  has_air_conditioning  automatic_car  \
708                        True     True                  True           True   
1641                       True     True                 False          False   
3440                       True     True                  True          False   

      has_getaround_connect  has_speed_regulator  winter_tires  
708          

---
Basic Pipeline

Il s'agit du même pipeline, on ne réécrit pas le code

---
Fit transform

In [129]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

---
Baseline

In [130]:
model_le.fit(X_train, Y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
# Predicts
Y_train_pred = model_le.predict(X_train)
Y_test_pred = model_le.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.6991247987385318
R2 score on test set :  0.7353146996921056


Le modèle a perdu en qualité avec autant de colonnes en moins.

In [133]:
# Visualize best coefficients

l_columns = []
for name, pipeline, features_list in preprocessor.transformers_: # for each tuple
    if name == 'num': 
        l_features = features_list 
    else: 
        l_features = pipeline.named_steps['encoder'].get_feature_names_out()
        l_features = list(l_features)
        l_features = np.char.replace(l_features, 'x0', features_list[0])
        l_features = np.char.replace(l_features, 'x1', features_list[1])
    l_columns.extend(l_features) 
        
coefs = pd.DataFrame(
    index=l_columns,
    data=model_le.coef_.transpose(),
    columns=["coefficients"]
    )

features_importance = coefs.sort_values(by='coefficients', key=abs)

fig = px.bar(
    features_importance,
    orientation='h',
    height=500
    )
fig.update_layout(
    showlegend=False, 
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
    )
fig.show()

---
# Random Forest

In [134]:
model_rfr_clustered = RandomForestRegressor()
model_rfr_clustered.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [135]:
Y_train_pred = model_rfr_clustered.predict(X_train)
Y_test_pred = model_rfr_clustered.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.9646306373800733
R2 score on test set :  0.7628990789483967


In [136]:
# Reduce overfitting - GridsearchCV

d_params = {
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [2, 4, 6],
    'min_samples_split': [2, 4, 6],
    'n_estimators': [10, 50, 100, 200, 300]
    }

gs_clustered = GridSearchCV(
        estimator=model_rfr,
        param_grid=d_params,
        cv=3,
        verbose=2
        )
gs_clustered.fit(X_train, Y_train)

Fitting 3 folds for each of 135 candidates, totalling 405 fits
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=10; total time=   0.0s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.4s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.4s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.4s
[CV] END max_depth=10, min_samples

,estimator,RandomForestRegressor()
,param_grid,"{'max_depth': [10, 15, ...], 'min_samples_leaf': [2, 4, ...], 'min_samples_split': [2, 4, ...], 'n_estimators': [10, 50, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [137]:
# Predicts
print("Best Parameters:", gs_clustered.best_params_)
print("Best Score:", gs_clustered.best_score_)

Y_train_pred = gs_clustered.predict(X_train)
Y_test_pred = gs_clustered.predict(X_test)
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Best Parameters: {'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 6, 'n_estimators': 300}
Best Score: 0.7334730678340292
R2 score on training set :  0.8993812426422712
R2 score on test set :  0.7608926220391677


Toujours overfitting